# H1 · `scripts/variants.py`

## What this file is for

Twenty controls a person may vary, and the one rule they all obey: **a legal value comes from
ISO's declaration for that jurisdiction**, never from the sample submission, a hardcoded list, or
what happened to work once. `Control.options` reads the domain table; `build` refuses a value that
is not in it.

Three callers read this one definition -- `scripts/breadth.py`, the `/tester` and `/tests` pages,
and `tests/verify_tester.py` -- so a second list of legal values would drift, and the drift would
look like a rating defect.

**Depends on:** `gl_engine` only. Nothing here renders anything, and nothing here imports `ui` --
the dependency runs one way, `ui -> variants -> gl_engine`.

## Its public surface

Generated from the module, so it can't drift.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))
sys.path.insert(0, str(Path.cwd().parent.parent / "scripts"))

import inspect
import variants as v

for name, obj in vars(v).items():
    if name.startswith("_") or getattr(obj, "__module__", None) != v.__name__:
        continue
    if inspect.isclass(obj):
        print(f"class {name}")
        for m, f in vars(obj).items():
            if m.startswith("_"):
                continue
            if isinstance(f, property):
                print(f"    .{m}  (property)")
            elif callable(f):
                try:
                    print(f"    .{m}{inspect.signature(f)}")
                except (TypeError, ValueError):
                    pass
    elif inspect.isfunction(obj):
        print(f"def {name}{inspect.signature(obj)}")

## The smallest thing that works

A configuration is a dict of control ids to values. `build` applies it to a jurisdiction's stored
base submission and hands back a payload ready to rate.

In [ ]:
from gl_engine.rating import Kernel

d = v.Declared("TX")
config = {"occurrence_limit": "2,000,000 CSL"}
payload = v.build(config, d)

print("configuration:", v.describe(config))
r = Kernel().rate(payload)
print("TX premium   :", r.premium)

## The interesting case

### The aggregate is half of the increased-limit key, and it is keyed on the occurrence limit

`general_aggregate` is its own control, but it is not a free choice: ISO declares which aggregates
are legal **for the occurrence limit in force**, and the set differs by state. Naming a figure like
"2,000,000" and running it in 51 states does not work -- it is undeclarable in some of them. The
control is keyed instead: given an occurrence limit, ask what aggregates ISO actually allows.

In [ ]:
occ = "2,000,000 CSL"
legal = d.aggregates_for(occ)
print(f"legal aggregates in TX at occurrence={occ}:")
print(" ", legal)

k = Kernel()
for agg in (legal[0], legal[-1]):
    p = v.build({"occurrence_limit": occ, "general_aggregate": agg}, d)
    r = k.rate(p)
    print(f"  aggregate={agg:<16} -> premium {r.premium}")

Leave the aggregate unset and `occurrence_limit`'s own applier derives it -- taking the **first**
value ISO declares legal, not a neutral default. `d.picks` records that a choice was made and what
the alternatives were, which is what `probe_no_op` reads to tell an inert *control* (nothing ISO
files would move it) from an inert *value* (we happened to pick one that doesn't, another would).

In [ ]:
d.picks.clear()
p2 = v.build({"occurrence_limit": occ}, d)
derived = p2["body"]["GeneralLiability"][0]["GeneralAggregateLimit"]
print("derived GeneralAggregateLimit:", derived)
print("recorded as a pick:", ("GeneralLiability", "GeneralAggregateLimit") in d.picks)

## What it refuses

Two different refusals, for two different reasons: a value outside the legal set, and a shape a
jurisdiction cannot express at all.

In [ ]:
try:
    v.build({"occurrence_limit": occ, "general_aggregate": "1,000,000 CSL"}, d)
except v.VariantError as exc:
    print("REFUSED (illegal value):", exc)

ak = v.Declared("AK")
print()
print("AK declares", len(ak.territories()), "prem/ops territory:", ak.territories())
try:
    v.build({"locations": 2}, ak)
except v.VariantError as exc:
    print("REFUSED (undeclarable shape):", exc)

## Try it yourself

1. `d.aggregates_for("500,000 CSL")` in TX -- how many legal aggregates at the lowest occurrence
   limit, versus the highest? Is the set the same size everywhere?
2. Find a jurisdiction where `NY`'s coverage-form rule applies -- `d.values(v.GL,
   "PremOpsProdsCoverageForm")` -- and confirm `Claims Made` is missing.
3. `v.union_options()` builds the same table `union_options` for every stored jurisdiction at once.
   How many states declare `Claims Made`, out of how many? [`H2 qa.py`](02-qa.ipynb) is the notebook
   that turns a configuration space like this into a bounded test plan.

In [ ]:
# your turn